# Instagram OAuth + Video Upload POC

Autenticação e upload de Reels via Meta API.

In [33]:
CLIENT_ID = "802226882950827"
CLIENT_SECRET = "6e37effcef17e0aff44e1f7bb0a5e8dc"
REDIRECT_URI = "https://localhost/callback"

In [34]:
AUTH_URL = "https://www.facebook.com/v19.0/dialog/oauth"

print(f"{AUTH_URL}?client_id={CLIENT_ID}&redirect_uri={REDIRECT_URI}&scope=instagram_basic,instagram_content_publish&response_type=code")

https://www.facebook.com/v19.0/dialog/oauth?client_id=802226882950827&redirect_uri=https://localhost/callback&scope=instagram_basic,instagram_content_publish&response_type=code


In [35]:
import requests
code = "AQCbfu5PqSeTErMvc4tXqKZUzJSKRDFMA_0yLztxzMxHaQfYV3na3OnNUPajAS9T4V_mpgVx7FiznfmQohjaw5mK318KwVHSISBR-ywvlD-oYNZ8dkAsbOiEpLvpoKF5xZmkFmJ-8SapE7DWtKmKPo4lOXhjqL4eG7j4O0fVROVL_4pFmGmB7sUEBXMkfwXdaIaTQbb5eYzLA3kVZvBqkCqvV2VAyPNH-Bl5Y9rpEv7rP-S_GXOJT6xGbxDGoXVU7o2Kc8R6y-Olm0KlyAo4CQtyDqb_VSBcb3C_NmhnckxkMXiCYLV7Z1iUh9DTOxFZx9nA591TsPdJmE1pVdQfXQ4_Rn88ZG8EQoGHG63T0PG6omgbis5uL6uVNKmBzr-SkbckK24JmlYugflEgyBmBjWpk3TKs4QiXSEsKnvpAj2oYQwg8tE2mSHeKBOoURlDRreTo6X9s2bG8hTugYLSj17-#_=_"

res = requests.get("https://graph.facebook.com/v19.0/oauth/access_token", params={
"client_id":CLIENT_ID,
"client_secret":CLIENT_SECRET,
"redirect_uri":REDIRECT_URI,
"code":code})

print(res.json())

{'error': {'message': 'This authorization code has expired.', 'type': 'OAuthException', 'code': 100, 'error_subcode': 36007, 'fbtrace_id': 'Ap7ZoM0QdT_Za0f01Mp-p3m'}}


In [36]:
# Criar container de vídeo
IG_USER_ID = "17841404274830804"
video_url = "https://samplelib.com/preview/mp4/sample-5s.mp4"

res = requests.post(f"https://graph.facebook.com/v19.0/{IG_USER_ID}/media", data={
"media_type":"REELS",
"video_url":video_url,
"caption":"Teste",
"access_token":"EAALZAnwBjNqsBRK5SI3FfnTAs54uO7gPdYFUJZAdVvVkxNC066GzxtrH0BJGWtYZCXx8SeREHoa34ohS5eguvfyyANYwH5CBZANFRI7IdMoYz1rds6yKZAyWGflWbXhKtnQwEuH9y6KnZBkLrQau66mMCLQ0ZCltgM6rSB3QVMT0L1RqZB6xS8OgmKKloJ1Yo8imrxoJv9tyrUZAap6GH2v7bhOZA01ax7u3Qa3Smp9UO8b5wv3ZAhifwodm7zH8GqshvsZAn8kzbQUOiJiyjFWx8s7srHmec9l8lLQr"})

print(res.json())

{'id': '18432897829142495'}


In [37]:
# Publicar

import time

# Pegar o creation_id retornado no passo anterior
container_id = res.json().get("id")

if not container_id:
    raise Exception("Erro: creation_id não retornado. Verifique a criação do container.")

# Aguardar processamento do container
while True:
    status_res = requests.get(
        f"https://graph.facebook.com/v19.0/{container_id}",
        params={
            "fields": "status_code",
            "access_token": "EAALZAnwBjNqsBRK5SI3FfnTAs54uO7gPdYFUJZAdVvVkxNC066GzxtrH0BJGWtYZCXx8SeREHoa34ohS5eguvfyyANYwH5CBZANFRI7IdMoYz1rds6yKZAyWGflWbXhKtnQwEuH9y6KnZBkLrQau66mMCLQ0ZCltgM6rSB3QVMT0L1RqZB6xS8OgmKKloJ1Yo8imrxoJv9tyrUZAap6GH2v7bhOZA01ax7u3Qa3Smp9UO8b5wv3ZAhifwodm7zH8GqshvsZAn8kzbQUOiJiyjFWx8s7srHmec9l8lLQr"
        }
    ).json()

    print("Status:", status_res)

    if status_res.get("status_code") == "FINISHED":
        break
    elif status_res.get("status_code") == "ERROR":
        raise Exception("Erro no processamento do container.")

    time.sleep(3)

# Publicar
publish_res = requests.post(
    f"https://graph.facebook.com/v19.0/{IG_USER_ID}/media_publish",
    data={
        "creation_id": container_id,
        "access_token": "EAALZAnwBjNqsBRK5SI3FfnTAs54uO7gPdYFUJZAdVvVkxNC066GzxtrH0BJGWtYZCXx8SeREHoa34ohS5eguvfyyANYwH5CBZANFRI7IdMoYz1rds6yKZAyWGflWbXhKtnQwEuH9y6KnZBkLrQau66mMCLQ0ZCltgM6rSB3QVMT0L1RqZB6xS8OgmKKloJ1Yo8imrxoJv9tyrUZAap6GH2v7bhOZA01ax7u3Qa3Smp9UO8b5wv3ZAhifwodm7zH8GqshvsZAn8kzbQUOiJiyjFWx8s7srHmec9l8lLQr"
    }
)

print(publish_res.json())

Status: {'status_code': 'IN_PROGRESS', 'id': '18432897829142495'}
Status: {'status_code': 'IN_PROGRESS', 'id': '18432897829142495'}
Status: {'status_code': 'IN_PROGRESS', 'id': '18432897829142495'}
Status: {'status_code': 'IN_PROGRESS', 'id': '18432897829142495'}
Status: {'status_code': 'IN_PROGRESS', 'id': '18432897829142495'}
Status: {'status_code': 'FINISHED', 'id': '18432897829142495'}
{'id': '17923448433280696'}
